# テーマA Phase A1b：A/B半差null検定 v0.2
> v0.2: 設定セルにCMBanomのclone（make_maskの依存）を復元
面除去後の鏡映反対称性がPR4検出器分割系統に由来しないことの検査（T側最後の系統検査）。

**ロジック**：
- 半和 (A+B)/2 → フルマップの反対称性（S⁺・軸・ℓ2–4抑制率0.10）を再現するはず
- 半差 (A−B)/2 → CMB信号は相殺。残る雑音＋分割系統の固定軸ℓ2–4対称成分パワーが，
  信号側で「欠けている」対称成分（null中央値の約90%）の何%かで**系統寄与を上限束縛**

所要：ダウンロード（2〜4ファイル・計1〜3GB）＋解析10分。

In [ ]:
# ---- 設定（マウント必須） ----
import os, sys, subprocess, time, json, urllib.request
IN_COLAB = os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('★Driveマウント失敗：再実行してください')
    OUTDIR='/content/drive/MyDrive/plane_mirror'
else: OUTDIR='plane_mirror_out'
os.makedirs(OUTDIR, exist_ok=True)
try: import healpy as hp
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','healpy']); import healpy as hp
import numpy as np, pandas as pd
if not os.path.isdir('CMBanom'):
    subprocess.run(['git','clone','--depth','1','https://github.com/LauraHerold/CMBanom.git'])
print('OUTDIR =', OUTDIR, '/ CMBanom OK:',
      os.path.exists('CMBanom/data/masks/com_mask_cutoff_0.9_nside_128.fits'))

In [ ]:
# ---- 検証済みモジュール ----
open('phase2_core.py','w').write(r'''# -*- coding: utf-8 -*-
"""Phase 2 較正基盤コア（v0.1）
- 処理構成マニフェスト（計画書v1.0 §3準拠）
- CRNマスター実現（synalm, seed 0..999, lmax=128）
- 統計①②③④の null 分布計算（チェックポイント/再開対応）
"""
import os, json, time
import numpy as np, healpy as hp

# ---------- 基本設定 ----------
LMAX_MASTER = 128
NSIM_FULL = 1000
FID_CL_FILE = 'CMBanom/data/real/COM_PowerSpect_CMB-base-plikHM-TTTEEE-lowl-lowE-lensing-minimum-theory_R3.01.txt'
COMMON_MASK_128 = 'CMBanom/data/masks/com_mask_cutoff_0.9_nside_128.fits'

def load_fid_cl(lmax=LMAX_MASTER):
    dat = np.loadtxt(FID_CL_FILE, skiprows=1)
    ll = np.arange(lmax + 1); cl = np.zeros(lmax + 1)
    n = min(lmax - 1, dat.shape[0])
    cl[2:2 + n] = dat[:n, 1] * 2 * np.pi / (ll[2:2 + n] * (ll[2:2 + n] + 1))
    return cl

# ---------- 伝達関数・マスク ----------
PIXWIN_CACHE = 'pixwin_cache'
def pixwin_pad(nside, lmax):
    fn = os.path.join(PIXWIN_CACHE, f'pixel_window_n{nside:04d}.fits')
    if os.path.exists(fn):
        from astropy.io import fits as _f
        with _f.open(fn) as h:
            pw = np.asarray(h[1].data['TEMPERATURE']).ravel()
    else:
        pw = hp.pixwin(nside)   # Colabでは通常経路（初回のみDL）
    return np.pad(pw, (0, max(0, lmax + 1 - len(pw))), mode='edge')[:lmax + 1]

def transfer(nside, smooth, lmax=LMAX_MASTER):
    """構成の伝達関数 b_ℓ p_ℓ。smooth ∈ {'planck','none','fix5deg'}"""
    pw = pixwin_pad(nside, lmax)
    if smooth == 'planck':
        fwhm_arcmin = 640.0 * 16.0 / nside
        return hp.gauss_beam(np.radians(fwhm_arcmin / 60.), lmax=lmax) * pw
    if smooth == 'fix5deg':
        return hp.gauss_beam(np.radians(5.0), lmax=lmax) * pw
    if smooth == 'none':
        return pw.copy()
    raise ValueError(smooth)

def dilate_mask(bad, nside, deg):
    """マスク（bad=True）を約deg度拡張（近傍膨張の反復）"""
    pixsize_deg = np.degrees(hp.nside2resol(nside))
    n_iter = max(1, int(np.ceil(deg / pixsize_deg)))
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _dilate_n(bad, nside, n_iter):
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _erode_n(bad, nside, n_iter):
    return ~_dilate_n(~bad, nside, n_iter)

def make_mask(nside, level):
    """level ∈ {'full','common','ext'} → bool（True=使用画素）
    ext = Planck 2015 XVI Table 12方式：共通マスクの拡散（銀河）成分のみを5°拡張し
          点源穴は拡張しない（補助マスク規定）。128で構成してから縮退。"""
    npix = hp.nside2npix(nside)
    if level == 'full':
        return np.ones(npix, bool)
    m128 = hp.read_map(COMMON_MASK_128)
    if level == 'common':
        return hp.ud_grade(m128, nside) >= 0.9
    if level == 'ext':
        bad128 = m128 < 0.5
        pix_deg = np.degrees(hp.nside2resol(128))          # ≈0.46°
        n_open = 2                                          # 開演算で点源(≲1°)を除去
        diffuse = _dilate_n(_erode_n(bad128, 128, n_open), 128, n_open)
        n5 = int(np.ceil(5.0 / pix_deg))                    # 5°膨張
        bad_ext = bad128 | _dilate_n(diffuse, 128, n5)
        return hp.ud_grade((~bad_ext).astype(float), nside) >= 0.9
    raise ValueError(level)

# ---------- CRNマスター実現 ----------
def gen_master_alm(outfile, nsim=NSIM_FULL, lmax=LMAX_MASTER):
    if os.path.exists(outfile):
        return np.load(outfile)['alms']
    cl = load_fid_cl(lmax)
    alms = np.empty((nsim, hp.Alm.getsize(lmax)), dtype=np.complex128)
    for s in range(nsim):
        np.random.seed(s)
        alms[s] = hp.synalm(cl, lmax=lmax)
    np.savez_compressed(outfile, alms=alms, lmax=lmax, nsim=nsim,
                        note='CRN master: fiducial PR3 bestfit, seeds 0..nsim-1')
    return alms

def config_maps(alms, nside, smooth, route='harmonic', lmax=LMAX_MASTER):
    """マスター実現→構成マップ群 (nsim, npix)"""
    bl = transfer(nside, smooth, lmax)
    npix = hp.nside2npix(nside)
    out = np.empty((alms.shape[0], npix))
    if route == 'harmonic':
        for s in range(alms.shape[0]):
            out[s] = hp.alm2map(hp.almxfl(alms[s], bl), nside)
    elif route == 'udgrade':   # 高解像度で実体化→画素平均（quadrature軸）
        nhi = min(4 * nside, 64)
        bl_hi = transfer(nside, smooth, lmax) / pixwin_pad(nside, lmax) * pixwin_pad(nhi, lmax)
        for s in range(alms.shape[0]):
            out[s] = hp.ud_grade(hp.alm2map(hp.almxfl(alms[s], bl_hi), nhi), nside)
    else:
        raise ValueError(route)
    return out

# ---------- 統計②：鏡映パリティ（マスク対応） ----------
class MirrorStat:
    def __init__(self, nside, mask):
        npix = hp.nside2npix(nside)
        vecs = np.array(hp.pix2vec(nside, np.arange(npix))).T
        R = np.empty((npix, npix), dtype=np.int32)
        for i in range(npix):
            n = vecs[i]
            refl = vecs - 2.0 * np.outer(vecs @ n, n)
            R[i] = hp.vec2pix(nside, refl[:, 0], refl[:, 1], refl[:, 2])
        self.R = R
        self.valid = mask[None, :] & mask[R]          # 双方が有効な対のみ
        self.cnt = np.maximum(self.valid.sum(axis=1), 1)
        self.mask = mask

    def min_S(self, mp, chunk=1024, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        mp = np.where(self.mask, mp, 0.0).astype(np.float32)
        nd = self.R.shape[0]
        Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
        for a in range(0, nd, chunk):
            Tr = mp[self.R[a:a + chunk]]
            V = self.valid[a:a + chunk]
            Sp[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] + Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
            Sm[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] - Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
        return float(Sp.min()), float(Sm.min())

# ---------- 統計①：R/D（QML / MASTER / naive fsky） ----------
def C_pm(cl_from2, lmax):
    ell = np.arange(2, lmax + 1)
    Dl = ell * (ell + 1.) / (2 * np.pi) * cl_from2[:lmax - 1]
    ev = (ell % 2 == 0)
    return Dl[ev].sum() / (lmax - 1), Dl[~ev].sum() / (lmax - 1)

def RD_traj(cl_from2, lmaxes):
    out = np.empty((len(lmaxes), 2))
    for i, L in enumerate(lmaxes):
        p, m = C_pm(cl_from2, L)
        out[i] = (p / m, p - m)
    return out

def lmax_est_of(nside):
    return int(min(40, 2.5 * nside))

# ---------- 統計③：多重極ベクトル（polyMV非依存・性質テスト済） ----------
from scipy.special import gammaln as _gln

def multipole_vectors(alm, lmax, ell):
    a = np.zeros(2*ell+1, dtype=complex)
    for m in range(0, ell+1):
        v = alm[hp.Alm.getidx(lmax, ell, m)]
        a[ell+m] = v
        if m: a[ell-m] = (-1)**m * np.conj(v)
    k = np.arange(2*ell+1)
    logC = _gln(2*ell+1) - _gln(k+1) - _gln(2*ell-k+1)
    c = np.sqrt(np.exp(logC)) * a
    roots = np.roots(c[::-1])
    th = 2*np.arctan(np.abs(roots)); ph = np.angle(roots) + np.pi
    v = np.stack([np.sin(th)*np.cos(ph), np.sin(th)*np.sin(ph), np.cos(th)], 1)
    v = np.where(v[:, 2:3] >= 0, v, -v)
    keep = []
    for i in range(len(v)):
        if not any(np.dot(v[i], v[j]) > 0.999 for j in keep): keep.append(i)
    return v[keep[:ell]]

def stat3_SQO(maps, mask, lmax_alm=8):
    """素朴カットスカイalm経路（第1弾で検証済みの規約）でS_QO"""
    vals = np.empty(maps.shape[0])
    fmask = mask.astype(float)
    for s in range(maps.shape[0]):
        alm = hp.map2alm(maps[s]*fmask, lmax=lmax_alm, iter=3)
        v2 = multipole_vectors(alm, lmax_alm, 2)
        v3 = multipole_vectors(alm, lmax_alm, 3)
        w2 = np.cross(v2[0], v2[1])
        w3 = [np.cross(v3[i], v3[j]) for i in range(3) for j in range(i+1, 3)]
        vals[s] = np.mean([abs(np.dot(w2, w)) for w in w3])
    return vals

# ---------- 実行系（チェックポイント） ----------
def null_dir(base, cfg_id):
    d = os.path.join(base, cfg_id); os.makedirs(d, exist_ok=True); return d

def true_varlvmap(lvmaps, lvmask, mean_lvmap):
    """正しい逆分散重み用の規格化分散（リポジトリ版get_varlvmapは定数(N-1)²/Nになるバグ）"""
    return np.where(lvmask == 1.,
                    np.mean((lvmaps - mean_lvmap) ** 2, axis=0) / np.maximum(mean_lvmap, 1e-30) ** 2,
                    1.)

def run_stat2(base, cfg_id, maps, nside, mask, chunk_ckpt=100, mondip=True):
    """②のnull分布（部分保存・再開対応）"""
    d = null_dir(base, cfg_id); fn = os.path.join(d, 'stat2.npz')
    done = 0; minSp = []; minSm = []
    if os.path.exists(fn):
        z = np.load(fn)
        if z['complete']: return
        minSp, minSm, done = list(z['minSp']), list(z['minSm']), int(z['done'])
    ms = MirrorStat(nside, mask)
    for s in range(done, maps.shape[0]):
        sp, sm = ms.min_S(maps[s], mondip=mondip)
        minSp.append(sp); minSm.append(sm)
        if (s + 1) % chunk_ckpt == 0 or s == maps.shape[0] - 1:
            np.savez(fn, minSp=minSp, minSm=minSm, done=s + 1,
                     complete=(s == maps.shape[0] - 1))
    return np.array(minSp), np.array(minSm)
''')
open('plane_mirror.py','w').write(r'''# -*- coding: utf-8 -*-
"""plane_mirror.py — テーマA: 鏡映反対称性の高速バッチ評価
MirrorStat（phase2_core）と厳密同一の定義で，方向走査をマップ束一括化。
検証済み（2026-08-15）: MirrorStatとの max相対差 1.2e-6（float32水準）。
計時: N16 33ms/マップ（10^5=55分）, N32 0.9s/マップ（10^4=2.5時間）。
"""
import numpy as np
import healpy as hp


class MirrorBatch:
    def __init__(self, ms):
        self.R, self.valid, self.cnt, self.mask = ms.R, ms.valid, ms.cnt, ms.mask

    def min_S_batch(self, maps, mondip=True, return_argmin=False):
        B = maps.shape[0]
        T = np.empty((maps.shape[1], B), np.float32)
        for b in range(B):
            m = maps[b]
            if mondip:
                m = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, m, hp.UNSEEN))))
            T[:, b] = np.where(self.mask, m, 0.0)
        nd = self.R.shape[0]
        minSp = np.full(B, np.inf, np.float32); minSm = np.full(B, np.inf, np.float32)
        argp = np.zeros(B, np.int32); argm = np.zeros(B, np.int32)
        for d in range(nd):
            v = self.valid[d]
            if not v.any():
                continue
            Tr = T[self.R[d]]
            Tv, Trv = T[v], Tr[v]
            Sp = np.einsum('ib,ib->b', 0.5 * (Tv + Trv), 0.5 * (Tv + Trv)) / self.cnt[d]
            Sm = np.einsum('ib,ib->b', 0.5 * (Tv - Trv), 0.5 * (Tv - Trv)) / self.cnt[d]
            better = Sp < minSp
            if return_argmin:
                argp = np.where(better, d, argp)
            minSp = np.where(better, Sp, minSp)
            better = Sm < minSm
            if return_argmin:
                argm = np.where(better, d, argm)
            minSm = np.where(better, Sm, minSm)
        if return_argmin:
            return minSp, minSm, argp, argm
        return minSp, minSm


def axis_lb(nside, d):
    """方向画素番号 → 銀経緯 (l, b) [deg]"""
    th, ph = hp.pix2ang(nside, int(d))
    return float(np.degrees(ph)), float(90.0 - np.degrees(th))


class FixedAxisMirror:
    """凍結軸 d* での対称成分解析（O(npix)/マップ）"""
    def __init__(self, ms, d_star):
        self.r = ms.R[d_star]
        self.v = ms.valid[d_star]
        self.cnt = ms.cnt[d_star]
        self.mask = ms.mask

    def S_plus(self, mp, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        return float(np.sum(self.v * s * s) / self.cnt)

    def S_plus_batch(self, maps, mondip=True):
        return np.array([self.S_plus(maps[b], mondip) for b in range(maps.shape[0])])

    def band_decompose(self, alm128, nside, transfer_fl, bands, mondip=True):
        """ソースalm(lmax128)を帯域分解し，帯域別S⁺と全帯域和・交差項を返す"""
        import numpy as _np
        LMAX = hp.Alm.getlmax(len(alm128))
        ell = _np.arange(LMAX + 1)
        out = {}
        s_parts = []
        for (l0, l1) in bands:
            w = ((ell >= l0) & (ell <= l1)).astype(float) * transfer_fl
            mb = hp.alm2map(hp.almxfl(alm128.copy(), w), nside)
            if mondip and l0 <= 1:
                pass
            T = _np.where(self.mask, mb, 0.0)
            if mondip:
                T = _np.where(self.mask,
                              _np.asarray(hp.remove_dipole(hp.ma(_np.where(self.mask, mb, hp.UNSEEN)))), 0.0)
            s = 0.5 * (T + T[self.r])
            s_parts.append(s)
            out[f'S{l0}_{l1}'] = float(_np.sum(self.v * s * s) / self.cnt)
        stot = _np.sum(s_parts, axis=0)
        out['S_sum_bands'] = float(_np.sum(self.v * stot * stot) / self.cnt)
        return out

    def exclusion_scan(self, mp, nside_scan=8, radius_deg=15.0, mondip=True):
        """半径radius_degの円盤を各走査位置で（鏡映相手も対称に）除外した際の
        ln S⁺ の変化 Δ(q) を返す（正＝除外で対称性が増える＝その領域が反対称の担い手）"""
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        base_num = np.sum(self.v * s * s)
        base = base_num / self.cnt
        npix_scan = hp.nside2npix(nside_scan)
        nside_map = hp.npix2nside(len(T))
        delta = np.zeros(npix_scan)
        for q in range(npix_scan):
            vec = hp.pix2vec(nside_scan, q)
            disc = hp.query_disc(nside_map, vec, np.radians(radius_deg))
            ex = np.zeros(len(T), bool); ex[disc] = True
            ex = ex | ex[self.r]                       # 対称除外
            v2 = self.v & ~ex
            c2 = max(v2.sum(), 1)
            S2 = np.sum(v2 * s * s) / c2
            delta[q] = np.log(S2 / base) if S2 > 0 else 0.0
        return delta, base


def with_mask(ms, mask):
    """R表を共有して別マスクのMirrorStat相当を作る（N32のR再構築20秒を節約）"""
    obj = type(ms).__new__(type(ms))
    obj.R = ms.R
    obj.mask = mask
    obj.valid = mask[None, :] & mask[ms.R]
    obj.cnt = np.maximum(obj.valid.sum(axis=1), 1)
    return obj


def scan_S(ms, mp, mondip=True):
    """1マップの全方向S±(n̂)地形を返す（軸の縮退・地形幅の解析用）"""
    if mondip:
        mp = np.asarray(hp.remove_dipole(hp.ma(np.where(ms.mask, mp, hp.UNSEEN))))
    T = np.where(ms.mask, mp, 0.0).astype(np.float32)
    nd = ms.R.shape[0]
    Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
    for d in range(nd):
        v = ms.valid[d]
        Tr = T[ms.R[d]]
        Sp[d] = np.sum(v * (0.5 * (T + Tr)) ** 2) / ms.cnt[d]
        Sm[d] = np.sum(v * (0.5 * (T - Tr)) ** 2) / ms.cnt[d]
    return Sp, Sm


def axis_sep_deg(nside, d1, d2):
    """2軸の分離角[deg]（鏡映面法線は±同一視 → 90°超は補角）"""
    v1 = np.array(hp.pix2vec(nside, int(d1)))
    v2 = np.array(hp.pix2vec(nside, int(d2)))
    ang = np.degrees(np.arccos(np.clip(abs(v1 @ v2), -1, 1)))
    return float(ang)
''')
import importlib, phase2_core as p2, plane_mirror as pm
for m in (p2,pm): importlib.reload(m)
print('OK')

In [ ]:
# ---- 候補ID走査：PR4 A/B分割compsepマップ ----
PLA='https://pla.esac.esa.int/pla/aio/product-action?MAP.MAP_ID='
CANDS=[]
for meth,ns in [('commander','4096'),('commander','2048'),('sevem','2048'),('sevem','4096')]:
    for tag in ['A','B']:
        for pat in [f'COM_CMB_IQU-{meth}_{ns}_R4.00_{tag}.fits',
                    f'COM_CMB_IQU-{meth}_{ns}_R4.00_split-{tag}.fits',
                    f'COM_CMB_IQU-{meth}_{ns}_R4.00_det{tag}.fits']:
            CANDS.append(pat)
# 予備: 2015 half-mission
for meth in ['smica','sevem','commander']:
    for hm in ['1','2']:
        CANDS.append(f'COM_CMB_IQU-{meth}_1024_R2.02_halfmission-{hm}.fits')
FOUND=[]
for fid in CANDS:
    try:
        req=urllib.request.Request(PLA+fid, headers={'Range':'bytes=0-1023'})
        with urllib.request.urlopen(req, timeout=30) as r:
            head=r.read(1024)
        if head[:6]==b'SIMPLE': FOUND.append(fid); print('HIT :', fid)
    except Exception: pass
print(f'\n発見 {len(FOUND)} 件。A/Bペアが揃った手法を使用します。')
PAIRS={}
for meth in ['commander','sevem','smica']:
    a=[f for f in FOUND if meth in f and ('_A.fits' in f or 'split-A' in f or 'detA' in f or 'halfmission-1' in f)]
    b=[f for f in FOUND if meth in f and ('_B.fits' in f or 'split-B' in f or 'detB' in f or 'halfmission-2' in f)]
    if a and b: PAIRS[meth]=(a[0],b[0]); print(f'{meth}: A={a[0]} / B={b[0]}')
if not PAIRS: print('★A/Bペア未発見——出力を添付してください。命名を調べて対処します')

In [ ]:
# ---- ダウンロード→ソース化（lmax128・5'解畳込み・単位/双極子処理） ----
LMAX=128
T_SRC = hp.gauss_beam(np.radians(1.0), lmax=LMAX)*p2.pixwin_pad(128, LMAX)
def to_source(fid):
    fn=os.path.basename(fid)
    if not os.path.exists(fn):
        print('DL:', fn); urllib.request.urlretrieve(PLA+fid, fn)
    m=hp.read_map(fn, field=0)
    if np.std(m)<5e-3: m=m*1e6
    m=np.asarray(hp.remove_dipole(hp.ma(m)))       # 太陽双極子・単極子を常に除去
    alm=hp.map2alm(m, lmax=LMAX)
    nsd=hp.npix2nside(len(m))
    bl_in=hp.gauss_beam(np.radians(5/60.),lmax=LMAX)*p2.pixwin_pad(nsd,LMAX)
    return hp.almxfl(alm, T_SRC/np.maximum(bl_in,1e-12))   # 共通ソース伝達へ
SRC={}
for meth,(fa_,fb_) in PAIRS.items():
    SRC[meth]=dict(A=to_source(fa_), B=to_source(fb_))
    print(meth, '準備OK')

In [ ]:
# ---- 検定：半和の再現性＋半差の系統上限 ----
axj=json.load(open(os.path.join(OUTDIR,'a1_consensus_axis.json')))
D_STAR=int(axj['axis_pix'])
mask=p2.make_mask(16,'common'); ms=p2.MirrorStat(16,mask)
fa=pm.FixedAxisMirror(ms,D_STAR)
mb=pm.MirrorBatch(ms)
bands_ref=pd.read_csv(os.path.join(OUTDIR,'a1_bands.csv'))
S24_nullmed=float(bands_ref[bands_ref.band=='S2_4'].S_null_med.iloc[0])
fl=p2.transfer(16,'planck',LMAX)/np.maximum(T_SRC,1e-12)
rows=[]
for meth,d in SRC.items():
    aS=0.5*(d['A']+d['B']); aD=0.5*(d['A']-d['B'])
    mS=hp.alm2map(hp.almxfl(aS.copy(),fl),16)
    mD=hp.alm2map(hp.almxfl(aD.copy(),fl),16)
    # 半和: S⁺・軸・帯域
    Sp,Sm,ap,am=mb.min_S_batch(mS[None,:],return_argmin=True)
    sep=pm.axis_sep_deg(16,D_STAR,int(ap[0]))
    bS=fa.band_decompose(hp.almxfl(aS.copy(),1.0/np.maximum(T_SRC,1e-12)) if False else aS,16,
                          p2.transfer(16,'planck',LMAX)/np.maximum(T_SRC,1e-12),[(2,4)])
    r24_sum=bS['S2_4']/S24_nullmed
    # 半差: 固定軸ℓ2-4対称成分
    bD=fa.band_decompose(aD,16,p2.transfer(16,'planck',LMAX)/np.maximum(T_SRC,1e-12),[(2,4)])
    deficit=S24_nullmed*(1-0.10)          # 信号側で欠けている対称成分
    frac=bD['S2_4']/deficit
    rows.append(dict(method=meth,S24_ratio_halfsum=r24_sum,axis_sep_halfsum_deg=sep,
                     S24_halfdiff=bD['S2_4'],deficit_ref=deficit,syst_frac=frac))
    print(f'{meth:10s} 半和: ℓ2-4比={r24_sum:.3f}（フル実測0.09-0.11と比較）軸ずれ={sep:.1f}°')
    print(f'{" ":10s} 半差: ℓ2-4対称パワー={bD["S2_4"]:.2f} μK² → 欠損対称成分の {100*frac:.2f}%')
pd.DataFrame(rows).to_csv(os.path.join(OUTDIR,'a1b_absplit.csv'),index=False)
print('\n判定基準（目安）: syst_frac ≪ 10% なら「分割系統では説明できない」——')
print('半和の再現（比0.09-0.11・軸ずれ≲4°）と併せて a1b_absplit.csv を添付してください')

## 完了後
`a1b_absplit.csv`とセル出力を添付してください。これがA1最終項目です。
以後：LiteBIRD予言の登録文書→A3（図表・執筆）へ。